# Assignment 3 — MM-IMDb M3

- Image-only: ResNet-50, fine-tuned.
- Text-only: BERT-base-uncased, fine-tuned.
- Simple fusion: frozen encoders, cached features, concatenation head.
- Improved fusion: Gated Multimodal Unit (GMU).
- Ablations: GMU with and without modality dropout, plus missing-modality evaluation.
- Analysis: per-class metrics, support/conflict cases, and per-class error counts.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "assignment3").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "assignment3" / "src"))

import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer

from utils.mmimdb import (
    build_label_matrix,
    build_splits,
    get_genres,
    load_mmimdb,
    make_image_loader,
    make_text_loader,
)
from models import (
    TEXT_MODEL_NAME,
    build_fusion_head,
    build_gmu,
    build_image_encoder,
    build_image_model,
    build_text_encoder,
    build_text_model,
)
from training_loop import (
    extract_image_features,
    extract_text_features,
    fit_fusion,
    fit_gmu,
    fit_image,
    fit_text,
    metrics_from_probabilities,
    run_feature_epoch,
    run_gmu_epoch,
    run_image_epoch,
    run_text_epoch,
)

SEED = 42
IMAGE_EPOCHS = 10
TEXT_EPOCHS = 5
FUSION_EPOCHS = 30
GMU_EPOCHS = 30
IMAGE_BATCH = 256
TEXT_BATCH = 64
FUSION_BATCH = 256
WORKERS = 8
LIMIT = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"

RESULTS_DIR = ROOT / "assignment3" / "results"
CHECKPOINT_DIR = ROOT / "assignment3" / "checkpoints"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

criterion = nn.BCEWithLogitsLoss()
print("device:", DEVICE)

device: cuda


In [2]:
dataset = load_mmimdb()
genres = get_genres(dataset)
labels = build_label_matrix(dataset, genres)
splits = build_splits(dataset, genres)

if LIMIT:
    splits = {name: indices[:LIMIT] for name, indices in splits.items()}
print("genres:", len(genres))
print("splits:", {name: len(indices) for name, indices in splits.items()})

genres: 23
splits: {'train': 15553, 'dev': 2595, 'test': 7805}


In [3]:
image_loaders = {
    name: make_image_loader(dataset, indices, labels, IMAGE_BATCH, name == "train", WORKERS)
    for name, indices in splits.items()
}
image_model = build_image_model(len(genres)).to(DEVICE)
image_history, image_best_epoch, image_thresholds = fit_image(
    image_model,
    image_loaders,
    DEVICE,
    epochs=IMAGE_EPOCHS,
    lr=3e-4,
    weight_decay=1e-4,
    amp=AMP,
    seed=SEED,
    checkpoint_path=CHECKPOINT_DIR / "image_resnet50.pt",
)

_, image_test_probs, image_test_targets = run_image_epoch(
    image_model, image_loaders["test"], criterion, DEVICE, AMP
)
image_metrics = metrics_from_probabilities(
    image_test_probs, image_test_targets, image_thresholds
)
print({key: image_metrics[key] for key in ("macro_f1", "micro_f1", "samples_f1", "exact_match")})

epoch 01/10 | train loss 0.3045 | val loss 0.2554 macro-F1 0.3105 | 35.2s


epoch 02/10 | train loss 0.2351 | val loss 0.2495 macro-F1 0.3391 | 15.2s


epoch 03/10 | train loss 0.2094 | val loss 0.2541 macro-F1 0.3492 | 65.7s


epoch 04/10 | train loss 0.1803 | val loss 0.2664 macro-F1 0.3508 | 27.5s


epoch 05/10 | train loss 0.1519 | val loss 0.2772 macro-F1 0.3546 | 72.0s


epoch 06/10 | train loss 0.1281 | val loss 0.2928 macro-F1 0.3475 | 20.0s


epoch 07/10 | train loss 0.1048 | val loss 0.3148 macro-F1 0.3442 | 18.6s


epoch 08/10 | train loss 0.0873 | val loss 0.3319 macro-F1 0.3377 | 24.3s


epoch 09/10 | train loss 0.0713 | val loss 0.3527 macro-F1 0.3400 | 14.3s


epoch 10/10 | train loss 0.0592 | val loss 0.3654 macro-F1 0.3440 | 13.5s


{'macro_f1': 0.3309516279438305, 'micro_f1': 0.4385189818977988, 'samples_f1': 0.4420046915014121, 'exact_match': 0.03728379244074311}


In [4]:
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_loaders = {
    name: make_text_loader(dataset, indices, labels, tokenizer, TEXT_BATCH, name == "train", WORKERS)
    for name, indices in splits.items()
}
text_model = build_text_model(len(genres)).to(DEVICE)
text_history, text_best_epoch, text_thresholds = fit_text(
    text_model,
    text_loaders,
    DEVICE,
    epochs=TEXT_EPOCHS,
    lr=2e-5,
    weight_decay=1e-4,
    amp=AMP,
    seed=SEED,
    checkpoint_path=CHECKPOINT_DIR / "text_bert.pt",
)

_, text_test_probs, text_test_targets = run_text_epoch(
    text_model, text_loaders["test"], criterion, DEVICE, AMP
)
text_metrics = metrics_from_probabilities(
    text_test_probs, text_test_targets, text_thresholds
)
print({key: text_metrics[key] for key in ("macro_f1", "micro_f1", "samples_f1", "exact_match")})

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


epoch 01/5 | train loss 0.3250 | val loss 0.2512 macro-F1 0.3925 | 27.1s


epoch 02/5 | train loss 0.2236 | val loss 0.2139 macro-F1 0.5056 | 35.2s


epoch 03/5 | train loss 0.1902 | val loss 0.1974 macro-F1 0.5381 | 29.8s


epoch 04/5 | train loss 0.1665 | val loss 0.1889 macro-F1 0.5708 | 34.4s


epoch 05/5 | train loss 0.1474 | val loss 0.1922 macro-F1 0.5782 | 43.2s


{'macro_f1': 0.5681865259940735, 'micro_f1': 0.6430431224737769, 'samples_f1': 0.6459993981390523, 'exact_match': 0.18193465727098015}


In [5]:
image_encoder = build_image_encoder().to(DEVICE)
text_encoder = build_text_encoder().to(DEVICE)

image_feature_loaders = {
    name: make_image_loader(dataset, indices, labels, IMAGE_BATCH, False, WORKERS)
    for name, indices in splits.items()
}
text_feature_loaders = {
    name: make_text_loader(dataset, indices, labels, tokenizer, TEXT_BATCH, False, WORKERS)
    for name, indices in splits.items()
}

image_feature_map = {}
text_feature_map = {}
fusion_features = {}
fusion_labels = {}
for name in splits:
    image_features, image_targets = extract_image_features(
        image_encoder, image_feature_loaders[name], DEVICE, AMP
    )
    text_features, text_targets = extract_text_features(
        text_encoder, text_feature_loaders[name], DEVICE, AMP
    )
    assert torch.equal(image_targets, text_targets)
    image_feature_map[name] = image_features
    text_feature_map[name] = text_features
    fusion_features[name] = torch.cat([image_features, text_features], dim=1)
    fusion_labels[name] = image_targets

fusion_head = build_fusion_head(
    input_dim=fusion_features["train"].shape[1],
    hidden_dim=512,
    num_labels=len(genres),
).to(DEVICE)
fusion_history, fusion_best_epoch, fusion_thresholds = fit_fusion(
    fusion_head,
    fusion_features["train"],
    fusion_labels["train"],
    fusion_features["dev"],
    fusion_labels["dev"],
    DEVICE,
    epochs=FUSION_EPOCHS,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=FUSION_BATCH,
    seed=SEED,
    checkpoint_path=CHECKPOINT_DIR / "fusion_mlp.pt",
)

test_loader = DataLoader(
    TensorDataset(fusion_features["test"], fusion_labels["test"]),
    batch_size=FUSION_BATCH,
    shuffle=False,
)
_, fusion_test_probs, fusion_test_targets = run_feature_epoch(
    fusion_head, test_loader, criterion, DEVICE
)
fusion_metrics = metrics_from_probabilities(
    fusion_test_probs, fusion_test_targets, fusion_thresholds
)
print({key: fusion_metrics[key] for key in ("macro_f1", "micro_f1", "samples_f1", "exact_match")})

epoch 01/30 | train loss 0.2697 | val macro-F1 0.4458


epoch 05/30 | train loss 0.1724 | val macro-F1 0.5728


epoch 10/30 | train loss 0.1548 | val macro-F1 0.5924


epoch 15/30 | train loss 0.1436 | val macro-F1 0.5981


epoch 20/30 | train loss 0.1349 | val macro-F1 0.6014


epoch 25/30 | train loss 0.1272 | val macro-F1 0.5967


epoch 30/30 | train loss 0.1185 | val macro-F1 0.5981


{'macro_f1': 0.5820475128425329, 'micro_f1': 0.6416209979809634, 'samples_f1': 0.6395240470256486, 'exact_match': 0.15259449071108264}


In [6]:
gmu_variants = {}
for name, dropout in [("gmu_no_dropout", 0.0), ("gmu_dropout", 0.2)]:
    print(f"\n=== {name} ===")
    gmu = build_gmu(num_labels=len(genres), hidden_dim=512, modality_dropout=dropout).to(DEVICE)
    history, best_epoch, thresholds = fit_gmu(
        gmu,
        image_feature_map["train"],
        text_feature_map["train"],
        fusion_labels["train"],
        image_feature_map["dev"],
        text_feature_map["dev"],
        fusion_labels["dev"],
        DEVICE,
        epochs=GMU_EPOCHS,
        lr=1e-3,
        weight_decay=1e-4,
        batch_size=FUSION_BATCH,
        modality_dropout=dropout,
        seed=SEED,
        checkpoint_path=CHECKPOINT_DIR / f"{name}.pt",
    )
    gmu_variants[name] = {
        "model": gmu,
        "history": history,
        "best_epoch": best_epoch,
        "thresholds": thresholds,
    }


=== gmu_no_dropout ===


epoch 01/30 | train loss 0.2701 | val macro-F1 0.4388


epoch 05/30 | train loss 0.1711 | val macro-F1 0.5774


epoch 10/30 | train loss 0.1541 | val macro-F1 0.5920


epoch 15/30 | train loss 0.1435 | val macro-F1 0.5973


epoch 20/30 | train loss 0.1347 | val macro-F1 0.5982


... (5 outputs left out)

epoch 15/30 | train loss 0.1730 | val macro-F1 0.6019


epoch 20/30 | train loss 0.1668 | val macro-F1 0.5987


epoch 25/30 | train loss 0.1642 | val macro-F1 0.6006


epoch 30/30 | train loss 0.1604 | val macro-F1 0.5980


In [7]:
gmu_metrics = {}
gmu_probabilities = {}
gmu_missing_metrics = {}

for name, variant in gmu_variants.items():
    gmu = variant["model"]
    thresholds = variant["thresholds"]
    test_loader = DataLoader(
        TensorDataset(
            image_feature_map["test"],
            text_feature_map["test"],
            fusion_labels["test"],
        ),
        batch_size=FUSION_BATCH,
        shuffle=False,
    )
    _, probabilities, targets = run_gmu_epoch(gmu, test_loader, criterion, DEVICE, AMP)
    gmu_metrics[name] = metrics_from_probabilities(probabilities, targets, thresholds)
    gmu_probabilities[name] = probabilities

    missing = {}
    for drop_mode in ("image", "text"):
        _, dropped_probs, dropped_targets = run_gmu_epoch(
            gmu, test_loader, criterion, DEVICE, AMP, drop_mode=drop_mode
        )
        missing[drop_mode] = metrics_from_probabilities(
            dropped_probs, dropped_targets, thresholds
        )
    gmu_missing_metrics[name] = missing

for name, metrics in gmu_metrics.items():
    print(name, {key: metrics[key] for key in ("macro_f1", "micro_f1", "samples_f1", "exact_match")})
for name, missing in gmu_missing_metrics.items():
    print(name, "image dropped:", missing["image"]["macro_f1"], "text dropped:", missing["text"]["macro_f1"])


gmu_no_dropout {'macro_f1': 0.5882329321555028, 'micro_f1': 0.6503325836244437, 'samples_f1': 0.645514338320232, 'exact_match': 0.1464445868033312}
gmu_dropout {'macro_f1': 0.5859376184472439, 'micro_f1': 0.6493247353692664, 'samples_f1': 0.6466650667291283, 'exact_match': 0.15618193465727098}
gmu_no_dropout image dropped: 0.5225993747676114 text dropped: 0.2854791307141783
gmu_dropout image dropped: 0.5403782398905512 text dropped: 0.28730693870983487


In [8]:
def case_stats(predictions_a, predictions_b, targets):
    correct_a = (predictions_a == targets).all(axis=1)
    correct_b = (predictions_b == targets).all(axis=1)
    return {
        "both_correct": int(np.sum(correct_a & correct_b)),
        "a_only": int(np.sum(correct_a & ~correct_b)),
        "b_only": int(np.sum(~correct_a & correct_b)),
        "both_wrong": int(np.sum(~correct_a & ~correct_b)),
    }


def per_class_error_counts(predictions, targets):
    false_positive = ((predictions == 1) & (targets == 0)).sum(axis=0)
    false_negative = ((predictions == 0) & (targets == 1)).sum(axis=0)
    return false_positive, false_negative


all_probabilities = {
    "image_only": image_test_probs,
    "text_only": text_test_probs,
    "simple_fusion": fusion_test_probs,
    **gmu_probabilities,
}
all_thresholds = {
    "image_only": image_thresholds,
    "text_only": text_thresholds,
    "simple_fusion": fusion_thresholds,
    **{name: variant["thresholds"] for name, variant in gmu_variants.items()},
}
all_predictions = {
    name: (probabilities >= all_thresholds[name][None, :]).astype(np.int32)
    for name, probabilities in all_probabilities.items()
}
all_metrics = {
    "image_only": image_metrics,
    "text_only": text_metrics,
    "simple_fusion": fusion_metrics,
    **gmu_metrics,
}

support_analysis = {}
for name in ("simple_fusion", "gmu_dropout", "gmu_no_dropout"):
    support_analysis[name] = case_stats(
        all_predictions[name],
        all_predictions["text_only"],
        fusion_labels["test"].numpy(),
    )

error_counts = {}
for name, predictions in all_predictions.items():
    fp, fn = per_class_error_counts(predictions, fusion_labels["test"].numpy())
    ranked = sorted(
        [
            {
                "genre": genres[index],
                "false_positive": int(fp[index]),
                "false_negative": int(fn[index]),
                "total": int(fp[index] + fn[index]),
            }
            for index in range(len(genres))
        ],
        key=lambda item: item["total"],
        reverse=True,
    )
    error_counts[name] = ranked[:10]

example_cases = {}
for name in ("simple_fusion", "gmu_dropout"):
    a = all_predictions[name]
    b = all_predictions["text_only"]
    correct_a = (a == fusion_labels["test"].numpy()).all(axis=1)
    correct_b = (b == fusion_labels["test"].numpy()).all(axis=1)
    masks = {
        "fusion_correct_text_wrong": correct_a & ~correct_b,
        "fusion_wrong_text_correct": ~correct_a & correct_b,
        "both_wrong": ~correct_a & ~correct_b,
        "both_correct": correct_a & correct_b,
    }
    example_cases[name] = {
        case: np.flatnonzero(mask)[:20].tolist()
        for case, mask in masks.items()
    }

np.savez(
    RESULTS_DIR / "m3_predictions.npz",
    targets=fusion_labels["test"].numpy(),
    **{f"pred_{name}": pred for name, pred in all_predictions.items()},
)
print("support analysis:", support_analysis)

support analysis: {'simple_fusion': {'both_correct': 583, 'a_only': 608, 'b_only': 837, 'both_wrong': 5777}, 'gmu_dropout': {'both_correct': 600, 'a_only': 619, 'b_only': 820, 'both_wrong': 5766}, 'gmu_no_dropout': {'both_correct': 573, 'a_only': 570, 'b_only': 847, 'both_wrong': 5815}}


In [9]:
model_rows = []
for name, model, metrics, history, best_epoch, thresholds in [
    ("Image-only ResNet-50", image_model, image_metrics, image_history, image_best_epoch, image_thresholds),
    ("Text-only BERT", text_model, text_metrics, text_history, text_best_epoch, text_thresholds),
    ("Simple fusion", fusion_head, fusion_metrics, fusion_history, fusion_best_epoch, fusion_thresholds),
] + [
    (name, variant["model"], gmu_metrics[name], variant["history"], variant["best_epoch"], variant["thresholds"])
    for name, variant in gmu_variants.items()
]:
    model_rows.append(
        {
            "model": name,
            "params": sum(p.numel() for p in model.parameters()),
            "best_epoch": int(best_epoch),
            "train_time_s": float(sum(history["epoch_time"])),
            "test_macro_f1": metrics["macro_f1"],
            "test_micro_f1": metrics["micro_f1"],
            "test_samples_f1": metrics["samples_f1"],
            "test_exact_match": metrics["exact_match"],
        }
    )

results_df = pd.DataFrame(model_rows)
results_df

,model,params,best_epoch,train_time_s,test_macro_f1,test_micro_f1,test_samples_f1,test_exact_match
0,Image-only ResNet-50,23555159,5,306.391155,0.330952,0.438519,0.442005,0.037284
1,Text-only BERT,109499927,5,169.684623,0.568187,0.643043,0.645999,0.181935
2,Simple fusion,1454103,24,37.755999,0.582048,0.641621,0.639524,0.152594
3,gmu_no_dropout,2896919,14,37.813045,0.588233,0.650333,0.645514,0.146445
4,gmu_dropout,2896919,27,51.571871,0.585938,0.649325,0.646665,0.156182


In [10]:
config = {
    "dataset": "MM-IMDb",
    "seed": SEED,
    "image_epochs": IMAGE_EPOCHS,
    "text_epochs": TEXT_EPOCHS,
    "fusion_epochs": FUSION_EPOCHS,
    "gmu_epochs": GMU_EPOCHS,
    "image_batch": IMAGE_BATCH,
    "text_batch": TEXT_BATCH,
    "fusion_batch": FUSION_BATCH,
    "limit": LIMIT,
    "amp": AMP,
    "device": str(DEVICE),
}
payload = {
    "config": config,
    "genres": genres,
    "splits": {name: len(indices) for name, indices in splits.items()},
    "results": model_rows,
    "gmu_missing_metrics": gmu_missing_metrics,
    "support_analysis": support_analysis,
    "error_counts": error_counts,
    "example_cases": example_cases,
    "per_class_f1": {name: metrics["per_class_f1"] for name, metrics in all_metrics.items()},
    "thresholds": {
        "image": image_thresholds.tolist(),
        "text": text_thresholds.tolist(),
        "simple_fusion": fusion_thresholds.tolist(),
        **{name: variant["thresholds"].tolist() for name, variant in gmu_variants.items()},
    },
    "history": {
        "image": image_history,
        "text": text_history,
        "simple_fusion": fusion_history,
        **{name: variant["history"] for name, variant in gmu_variants.items()},
    },
}
with open(RESULTS_DIR / "m3_results.json", "w") as handle:
    json.dump(payload, handle, indent=2)
results_df.to_csv(RESULTS_DIR / "m3_results.csv", index=False)
with open(RESULTS_DIR / "m3_error_analysis.json", "w") as handle:
    json.dump(
        {
            "support_analysis": support_analysis,
            "error_counts": error_counts,
            "example_cases": example_cases,
        },
        handle,
        indent=2,
    )
print("saved", RESULTS_DIR / "m3_results.json")

saved /raid/home/an.nv/main/deep-learning/CO3133-DeepLearning/assignment3/results/m3_results.json
